# 02 · Live USB webcam pose test

Run the cells **one at a time**. Your computer's browser captures the Logitech USB camera; Python inside this devcontainer runs the downloaded selected **MediaPipe, YOLO Pose, MoveNet, RTMPose, or RTMO** model. This is human pose detection, not general object detection. No USB passthrough is required. Alternative models need the optional packages below.

Frames travel through a forwarded port and are processed in memory; video is not recorded. Detection summaries (timestamps, pose status, ankle coordinates and native scores) are saved to `outputs/webcam_detection_*.txt`. Keep the port private. The preview shows the exact submitted frame plus its pose, so network/inference delay is visible.

## 1. Prepare your computer

1. Plug in the Logitech Brio and close other apps using it.
2. Open this notebook in VS Code and select the `/usr/local/bin/python` kernel.
3. Use a browser running **on the computer connected to the camera**.
4. Start with one person, good lighting, and the entire body and feet visible.


See [the model guide](../docs/MODELS.md) for model purposes, exact sources, lower-body limitations, score/log definitions and experiment results.


In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone
import secrets
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

import cv2
import mediapipe as mp
import numpy as np

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "src" / "walk_analysis").is_dir())
from walk_analysis.pose_models import MODEL_SPECS, PoseDetector, model_status
MODEL_STATUS = model_status(PROJECT_ROOT / "models")
AVAILABLE_MODELS = [name for name, state in MODEL_STATUS.items() if state["available"]]
if not AVAILABLE_MODELS:
    raise RuntimeError("No usable models. Follow the install/download instructions below.")
for name, state in MODEL_STATUS.items():
    print(f"{state['label']}: {state['reason'] or 'ready'}")


## 2. Install optional backends, download, and configure

For the five additional test options, run these commands **once in the devcontainer terminal**. CPU-only PyTorch avoids CUDA downloads:

```bash
python -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
python -m pip install -r requirements-pose.txt
python -m pip install --no-deps rtmlib==0.0.16
python -m walk_analysis download-model --model all
```

RTMLib is installed without dependencies because its package requests GUI OpenCV; this project supplies headless OpenCV and the required ONNX runtime separately. These optional packages do not change the default MediaPipe workflow. To download only the five additions, use `--model alternatives`; use `--model mediapipe` for just the three MediaPipe bundles. Model downloads need internet; live inference uses local weights.

**Restart the kernel after installing**, then run step 1 again. Missing weights or packages appear as disabled browser choices. The five additions are a practical selection of commonly used pose systems, not a measured popularity ranking:

| Browser choice | Implementation | Output |
|---|---|---|
| YOLOv8n Pose | Ultralytics, CPU | COCO 17 points |
| YOLO11n Pose | Ultralytics, CPU | COCO 17 points |
| MoveNet Thunder | Google LiteRT, CPU | COCO 17 points |
| RTMPose-m + YOLOX-m | RTMLib / ONNX Runtime, CPU | COCO 17 points |
| RTMO-m | RTMLib / ONNX Runtime, CPU | COCO 17 points |
| MediaPipe Lite / Full / Heavy | MediaPipe, CPU | 33 points including heels/toes |

Sources: [Ultralytics](https://docs.ultralytics.com/tasks/pose/), [MoveNet](https://www.tensorflow.org/hub/tutorials/movenet), [RTMLib](https://github.com/Tau-J/rtmlib). Upstream licenses are linked in those repositories; YOLO uses Ultralytics' AGPL/enterprise licensing.

Start at 640 × 480 and 10 requested submissions/second. Models may internally resize the frame differently. The browser reports actual camera resolution. Only one model runs at a time; switching can pause while loading. Change the settings below and restart step 4 to apply them.


In [ ]:
PORT = 8765
CAMERA_WIDTH = 640
CAMERA_HEIGHT = 480
TARGET_FPS = 10
VISIBILITY = 0.5
DEFAULT_MODEL = "lite" if "lite" in AVAILABLE_MODELS else AVAILABLE_MODELS[0]


## 3. Create the browser capture page

Click **Enable camera**, allow browser camera access, select the Logitech device, then click **Start pose test**. Green lines show the body; orange lines show hips and legs. COCO models have no heel/toe points, so those lines are omitted. The status displays pose coverage for this session and ankle score. A visible skeleton alone does not validate step counts.

The frame includes a readable dark text panel with UTC time, frame number, pose coverage, inference time, and ankle score. The last completed frame and skeleton remain visible while the next frame is processed, removing the redraw flash. Each processed frame also gets a line in the text log.


In [ ]:
HTML_TEMPLATE = '<!doctype html><meta charset="utf-8"><title>Live webcam pose test</title>\n<style>body{font:16px system-ui;max-width:960px;margin:24px auto;padding:0 16px;background:#18212b;color:#eee}button,select{padding:10px;margin:4px}canvas{width:100%;background:#000}video{display:none}#status{white-space:pre-wrap}</style>\n<h1>Live webcam pose test</h1><p>Camera → browser → Python in devcontainer → pose overlay. Frames are not recorded.</p>\n<button id="enable">Enable camera</button><select id="devices" aria-label="Camera"></select><label>Pose model <select id="model" aria-label="Pose model"></select></label><button id="start" disabled>Start pose test</button><button id="stop">Stop camera</button>\n<p id="status">Enable the camera first.</p><video id="video" autoplay muted playsinline></video><canvas id="canvas"></canvas>\n<script>\nconst cfg=__CONFIG__, video=document.querySelector(\'#video\'), canvas=document.querySelector(\'#canvas\'), ctx=canvas.getContext(\'2d\'), status=document.querySelector(\'#status\'), devices=document.querySelector(\'#devices\');\nconst capture=document.createElement(\'canvas\'), captureCtx=capture.getContext(\'2d\');\nconst modelSelect=document.querySelector(\'#model\');\nfor(const [name,info] of Object.entries(cfg.catalog)){const option=new Option(info.label+(info.available?\'\':\' (\'+info.reason+\')\'),name);option.disabled=!info.available;modelSelect.add(option);}modelSelect.value=cfg.defaultModel;\nmodelSelect.onchange=()=>{count=found=0;status.textContent=\'Selected \'+modelSelect.value+\'. Loading on next frame; first frame can be slower.\';};\nlet stream=null, running=false, busy=false, count=0, found=0;\nfunction stop(){running=false;if(stream)stream.getTracks().forEach(t=>t.stop());stream=null;document.querySelector(\'#start\').disabled=true;}\nasync function camera(id){stop();if(!navigator.mediaDevices)throw Error(\'Camera requires localhost or HTTPS. Open in your computer’s external browser.\');stream=await navigator.mediaDevices.getUserMedia({audio:false,video:{width:{ideal:cfg.width},height:{ideal:cfg.height},...(id?{deviceId:{exact:id}}:{})}});video.srcObject=stream;await video.play();canvas.width=video.videoWidth;canvas.height=video.videoHeight;capture.width=canvas.width;capture.height=canvas.height;ctx.drawImage(video,0,0);document.querySelector(\'#start\').disabled=false;status.textContent=\'Camera ready: \'+canvas.width+\' × \'+canvas.height+\'. Select Logitech, then start.\';}\ndocument.querySelector(\'#enable\').onclick=async()=>{try{await camera();const active=stream.getVideoTracks()[0].getSettings().deviceId;devices.replaceChildren();for(const d of await navigator.mediaDevices.enumerateDevices()){if(d.kind===\'videoinput\'){const o=new Option(d.label||\'Camera\',d.deviceId);devices.add(o);}}devices.value=active;}catch(e){status.textContent=e.name+\': \'+e.message;}};\ndevices.onchange=async()=>{try{await camera(devices.value);}catch(e){status.textContent=e.message;}};\ndocument.querySelector(\'#stop\').onclick=()=>{stop();status.textContent=\'Stopped. Camera released.\';};\nconst edges=[[11,12],[11,23],[12,24],[23,24],[11,13],[13,15],[12,14],[14,16],[23,25],[25,27],[27,29],[29,31],[27,31],[24,26],[26,28],[28,30],[30,32],[28,32]];\nasync function loop(){if(!running)return;busy=true;const t=performance.now();try{captureCtx.drawImage(video,0,0);const blob=await new Promise(r=>capture.toBlob(r,\'image/jpeg\',0.8));const res=await fetch(\'/frame\',{method:\'POST\',headers:{\'X-Camera-Token\':cfg.token,\'X-Pose-Model\':modelSelect.value},body:blob});const data=await res.json();if(!res.ok)throw Error(data.error||res.statusText);if(!running||data.model!==modelSelect.value)return;ctx.drawImage(capture,0,0);count++;const p=data.landmarks;if(p.length)found++;const visible=i=>p[i]&&p[i].v>=cfg.visibility;for(const [a,b] of edges){if(visible(a)&&visible(b)){ctx.strokeStyle=a>=23?\'#ffad42\':\'#63e899\';ctx.lineWidth=3;ctx.beginPath();ctx.moveTo(p[a].x*canvas.width,p[a].y*canvas.height);ctx.lineTo(p[b].x*canvas.width,p[b].y*canvas.height);ctx.stroke();}}for(let i=0;i<p.length;i++){if(visible(i)){ctx.fillStyle=i>=23?\'#ffad42\':\'#63e899\';ctx.beginPath();ctx.arc(p[i].x*canvas.width,p[i].y*canvas.height,4,0,Math.PI*2);ctx.fill();}}const lines=[data.model.toUpperCase()+\' | \'+data.time+\' | \'+(p.length?\'POSE DETECTED\':\'NO POSE\'), data.score_kind+\' | frame \'+data.frame+\' | coverage \'+Math.round(found/count*100)+\'% | inference \'+data.inference_ms+\' ms\', p.length?\'Ankle score: L \'+p[27].v.toFixed(2)+\'  R \'+p[28].v.toFixed(2):\'Keep full body and feet visible\'];ctx.save();const fontSize=Math.max(14,Math.round(canvas.width/40));ctx.font=fontSize+\'px monospace\';const lineHeight=fontSize+6;ctx.fillStyle=\'rgba(0,0,0,0.8)\';ctx.fillRect(0,0,canvas.width,lines.length*lineHeight+16);ctx.fillStyle=\'#ffffff\';ctx.textBaseline=\'top\';lines.forEach((line,i)=>ctx.fillText(line,8,8+i*lineHeight,canvas.width-16));ctx.restore();status.textContent=data.model.toUpperCase()+\' | \'+(p.length?\'POSE DETECTED\':\'NO POSE — check full-body framing\')+\' | coverage \'+found+\'/\'+count+\' (\'+Math.round(found/count*100)+\'%) | round trip \'+Math.round(performance.now()-t)+\' ms\'+(p.length?\' | ankle score L \'+p[27].v.toFixed(2)+\' R \'+p[28].v.toFixed(2):\'\');}catch(e){running=false;status.textContent=\'Test stopped: \'+e.message+\'. Stop camera or restart the server.\';}finally{busy=false;if(running)setTimeout(loop,Math.max(0,1000/cfg.fps-(performance.now()-t)));}}\ndocument.querySelector(\'#start\').onclick=()=>{if(!stream||running||busy)return;count=found=0;running=true;loop();};window.addEventListener(\'pagehide\',stop);\n</script>'


## 4. Start Python inference and forward the port

After running this cell:

1. Open VS Code's **Ports** panel (Terminal → Ports).
2. Click **Forward a Port**, enter `8765` (or your configured port), and keep visibility **Private**.
3. Choose **Open in Browser**, not the embedded preview. Use the forwarded local address, usually `http://localhost:8765`. For a remote workspace, use its private HTTPS forwarded URL.
4. Enable the camera and select Logitech. Your browser or operating system may ask for camera permission.

The browser needs localhost or HTTPS for camera access ([browser documentation](https://developer.mozilla.org/en-US/docs/Web/API/MediaDevices/getUserMedia)). Python uses VIDEO mode with increasing timestamps ([MediaPipe documentation](https://ai.google.dev/edge/mediapipe/solutions/vision/pose_landmarker/python)). Requests run sequentially, which avoids a queue of stale frames. Only one browser test should run at a time.


In [ ]:
# Safe to rerun: shut down the previous server and model first.
if "server" in globals():
    server.shutdown()
    server.server_close()
    server_thread.join()
    with inference_lock:
        detector.close()

inference_lock = threading.Lock()
token = secrets.token_urlsafe(32)
page = HTML_TEMPLATE.replace("__CONFIG__", json.dumps(dict(
    width=CAMERA_WIDTH, height=CAMERA_HEIGHT, fps=TARGET_FPS,
    visibility=VISIBILITY, token=token, models=AVAILABLE_MODELS, catalog=MODEL_STATUS, defaultModel=DEFAULT_MODEL))).encode()
def create_detector(name):
    return PoseDetector(name, PROJECT_ROOT / "models")

active_model = DEFAULT_MODEL
detector = create_detector(active_model)
last_timestamp = -1
frame_number = 0
LOG_DIR = PROJECT_ROOT / "outputs"
LOG_DIR.mkdir(exist_ok=True)
LOG_PATH = LOG_DIR / (datetime.now(timezone.utc).strftime("webcam_detection_%Y%m%d_%H%M%S_%f") + ".txt")
LOG_PATH.write_text(
    "Webcam pose detection log (UTC)\n"
    f"Available models: {AVAILABLE_MODELS} | drawing visibility threshold: {VISIBILITY}\n"
    "Each line is one processed frame. Native scores differ between families and are not accuracy.\n\n"
)


class CameraHandler(BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass  # Avoid printing one log line per frame.

    def reply(self, status, content, mime="application/json"):
        self.send_response(status)
        self.send_header("Content-Type", mime)
        self.send_header("Content-Length", str(len(content)))
        self.send_header("Cache-Control", "no-store")
        self.end_headers()
        self.wfile.write(content)

    def do_GET(self):
        if self.path != "/":
            return self.reply(404, b'{}')
        self.reply(200, page, "text/html; charset=utf-8")

    def do_POST(self):
        global last_timestamp, frame_number, active_model, detector
        if self.path != "/frame":
            return self.reply(404, b'{}')
        if self.headers.get("X-Camera-Token") != token:
            return self.reply(403, b'{"error":"Invalid session token; reload page"}')
        selected_model = self.headers.get("X-Pose-Model", DEFAULT_MODEL)
        if selected_model not in AVAILABLE_MODELS:
            return self.reply(400, b'{"error":"Model unavailable; download it and restart step 4"}')
        try:
            size = int(self.headers.get("Content-Length", 0))
            if not 0 < size <= 5_000_000:
                return self.reply(413, b'{"error":"Frame too large or empty"}')
            raw = self.rfile.read(size)
            frame = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR)
            if frame is None:
                raise ValueError("Invalid JPEG frame")
            with inference_lock:
                if selected_model != active_model:
                    replacement = create_detector(selected_model)
                    detector.close()
                    detector = replacement
                    active_model = selected_model
                    last_timestamp = -1
                timestamp = max(last_timestamp + 1, time.monotonic_ns() // 1_000_000)
                last_timestamp = timestamp
                started = time.perf_counter()
                landmarks = detector.detect(frame, timestamp)
                inference_ms = round((time.perf_counter() - started) * 1000, 1)
                frame_number += 1
                now = datetime.now(timezone.utc).isoformat(timespec="milliseconds")
                state = "POSE DETECTED" if landmarks else "NO POSE"
                detail = ""
                if landmarks:
                    detail = (f" | ankle score L={landmarks[27].visibility:.2f}"
                              f" R={landmarks[28].visibility:.2f}"
                              f" | ankle xy L=({landmarks[27].x:.3f}, {landmarks[27].y:.3f})"
                              f" R=({landmarks[28].x:.3f}, {landmarks[28].y:.3f})")
                    detail += " | leg scores " + " ".join(
                        f"{label}={landmarks[index].visibility:.2f}"
                        for label, index in [("hipL",23),("hipR",24),("kneeL",25),("kneeR",26)])
                with LOG_PATH.open("a", encoding="utf-8") as log:
                    log.write(f"{now} | model {active_model:<5} | frame {frame_number:06d} | {state:<13}"
                              f" | inference {inference_ms:6.1f} ms | score_kind={detector.score_kind}{detail}\n")
                payload = dict(score_kind=detector.score_kind, model=active_model, time=now, frame=frame_number, inference_ms=inference_ms,
                               landmarks=[dict(x=p.x, y=p.y, v=p.visibility) for p in landmarks])
            self.reply(200, json.dumps(payload).encode())
        except Exception as exc:
            self.reply(400, json.dumps(dict(error=str(exc))).encode())

try:
    server = ThreadingHTTPServer(("0.0.0.0", PORT), CameraHandler)
except OSError:
    detector.close()
    raise
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()
print(f"Server running. Forward port {PORT} privately and open it in your computer's browser.")
print("Detection text log:", LOG_PATH)


## 5. Compare the models and inspect logs

Use the browser's **Pose model** selector. Switching resets the coverage counter and discards in-flight results from the previous model. Keep one person in view: multi-person backends select the strongest candidate, without persistent identity tracking.

1. Test each model for at least 20 seconds with full body and feet visible, using the same lighting, camera distance and movements.
2. Repeat with your intended lower-body framing. All supplied checkpoints are full-body pose systems; cropped legs can fail with every model.
3. Open the printed `outputs/webcam_detection_*.txt` path. Each line records model, time, pose status, end-to-end Python processing time (including preprocessing/person detection), native score type, ankle coordinates, and hip/knee scores. Timing excludes model loading, network delay, and browser drawing.
4. Visually check knees/ankles against the image. For a controlled comparison, later process the same recorded video with every model.

**Scores and coverage:** MediaPipe reports visibility; other backends report keypoint confidence. These scores are not calibrated across models, so 0.5 does not mean the same thing for all of them. The drawing threshold applies to the raw native score. RTMPose scores can exceed 1. COCO models map only their 17 available points into the display; missing heels/toes remain hidden. For non-MediaPipe models, this application accepts a returned pose only if at least four keypoints score >= 0.2, in addition to any backend person-detection threshold. Thus pose coverage is an application diagnostic, not a standardized accuracy benchmark.

**Disappearing skeleton:** NO POSE means no accepted candidate. POSE DETECTED with missing lines means those joints fell below the drawing threshold or outside the image. The last completed overlay remains visible during the next inference; stale poses are not drawn onto new images when detection fails. Improve framing and lighting first, then optionally experiment with `VISIBILITY = 0.3`.

**Troubleshooting:** Use localhost or private HTTPS in an external browser; allow camera permission and close other camera apps. If the server is unreachable, verify the kernel is running and port is forwarded. After downloading/installing new models, rerun step 1 and step 4, then reload the browser. A failed model load appears in the browser status; select an available model and press Start again. If CPU processing is slow, reduce TARGET_FPS. These tests do not calculate cadence.

## 6. Stop and release resources

Click **Stop camera** first, then run this cell. Do not use Run All: this final cell stops the server immediately.


In [ ]:
if "server" in globals():
    server.shutdown()
    server.server_close()
    server_thread.join()
    with inference_lock:
        detector.close()
    del server
    print("Server stopped and model closed.")
else:
    print("No server is running.")
